In [ ]:
%matplotlib inline
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.calibration import calibration_curve
from src.metrics import delong_test, wilcoxon_paired, bonferroni
RES = Path("../results")
PASS = "defaults"   # change to "tuned" and re-execute the cells for second pass
df = pd.read_csv(RES / f"per_fold_{PASS}.csv")
preds = pd.read_parquet(RES / f"predictions_{PASS}.parquet")
models = list(df["model"].unique())
CHAMPION = "woe_logit"

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
for m in models:
    sub = preds[preds.model == m]
    fpr, tpr, _ = roc_curve(sub["y"], sub["proba"])
    ax.plot(fpr, tpr, label=m, lw=1.5)
ax.plot([0, 1], [0, 1], "k--", lw=0.5)
ax.set(xlabel="FPR", ylabel="TPR", title=f"ROC overlay (10-fold concat, {PASS})")
ax.legend(loc="lower right", fontsize=8)
fig.savefig(RES / "figures" / f"roc_overlay_{PASS}.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
for m in models:
    sub = preds[preds.model == m]
    prob_true, prob_pred = calibration_curve(sub["y"], sub["proba"], n_bins=10, strategy="quantile")
    ax.plot(prob_pred, prob_true, marker="o", label=m, lw=1.2)
ax.plot([0, 1], [0, 1], "k--", lw=0.5)
ax.set(xlabel="Predicted P(default)", ylabel="Observed P(default)", title=f"Reliability ({PASS})")
ax.legend(fontsize=8)
fig.savefig(RES / "figures" / f"reliability_{PASS}.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
data = [df[df.model == m]["auc"].values for m in models]
ax.boxplot(data, labels=models, vert=True)
ax.set(ylabel="AUC", title=f"Per-fold AUC distribution ({PASS})")
plt.xticks(rotation=30, ha="right")
fig.savefig(RES / "figures" / f"auc_boxplot_{PASS}.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
ch = preds[preds.model == CHAMPION].sort_values(["fold_idx", "test_idx"])
y_all = ch["y"].values
p_ch = ch["proba"].values

rows = []
for m in models:
    if m == CHAMPION: continue
    sub = preds[preds.model == m].sort_values(["fold_idx", "test_idx"])
    p_m = sub["proba"].values
    z, p = delong_test(y_all, p_m, p_ch)
    diff = roc_auc_score(y_all, p_m) - roc_auc_score(y_all, p_ch)
    se = abs(diff / z) if z != 0 else 0.05
    rows.append((m, diff, diff - 1.96 * se, diff + 1.96 * se, p))
fdf = pd.DataFrame(rows, columns=["model", "diff", "lo", "hi", "p"])

fig, ax = plt.subplots(figsize=(8, max(3, 0.5 * len(fdf))))
ax.errorbar(fdf["diff"], range(len(fdf)),
            xerr=[fdf["diff"] - fdf["lo"], fdf["hi"] - fdf["diff"]],
            fmt="o", color="black")
ax.axvline(0, color="grey", ls="--")
ax.set_yticks(range(len(fdf)))
ax.set_yticklabels([f"{r.model} (p={r.p:.3f})" for _, r in fdf.iterrows()])
ax.set_xlabel("AUC difference vs woe_logit champion")
ax.set_title(f"Forest plot of paired AUC diffs ({PASS})")
fig.savefig(RES / "figures" / f"forest_{PASS}.png", dpi=150, bbox_inches="tight")
plt.show()
fdf

In [ ]:
rt = df.groupby("model")["runtime_s"].agg(["mean", "sum"]).round(2)
rt.columns = ["mean_s_per_fold", "total_s"]
rt.sort_values("total_s", ascending=False)

In [ ]:
from IPython.display import Markdown, display
display(Markdown((RES / f"summary_{PASS}.md").read_text()))